## Set the experiment to process results for

In [40]:
import os
class settings:
  def __init__(self):
    self.dataset = "INA"
    self.iter=10
    self.results_dir="experiment_e2e100"
    self.methods = [['Med2Vec'], ['Dipole'], ['BiPadLSTM'], ['GRU']]  # for end-to-end experiment
    #self.methods = [['STATIC'],['STATIC', 'BINARY'],['STATIC', 'FREQUENCY'],['STATIC', 'EVENT-CNT'], 
    #            ['STATIC', 'TF-IDF'],['STATIC', 'BOW'],['STATIC', 'BiPadLSTM'],['STATIC', 'Med2Vec'],['STATIC', 'Dipole'],
    #            ['STATIC', 'CEHR-BERT'],['STATIC', 'DOME'],['STATIC', 'GRU'],]

args = settings()

## Load results

In [41]:
import pickle
# Caricamento dei risultati
results = {"+".join(m): {} for m in args.methods}

for i in range(args.iter):
    for m in args.methods:
        meth = "+".join(m)
        path = (
            f"{args.results_dir}/results_{args.dataset}_"
            f"{'_'.join(m)}_ITER{i}.pkl"
        )
        with open(path, "rb") as f:
            results[meth][i] = pickle.load(f)

## Compute metrics meand, standard deviation and CI intervals (with boost)

In [42]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    roc_auc_score,
)


METRIC_NAMES = [
    "AUC", "AUPRC", "F1", "PPV", "Sensitivity",
    "Specificity", "NPV", "MCC", "Accuracy", "Brier",
]


def metric_values(y, p, threshold=0.30):
    """Metrics in a repetition, one value per subject."""
    pred = (p >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(
        y, pred, labels=[0, 1]
    ).ravel()

    return np.array([
        roc_auc_score(y, p),
        average_precision_score(y, p),
        f1_score(y, pred, zero_division=0),
        precision_score(y, pred, zero_division=0),
        tp / (tp + fn),                         # Sensitivity
        tn / (tn + fp),                         # Specificity
        tn / (tn + fn) if (tn + fn) else 0.0,  # NPV
        matthews_corrcoef(y, pred),
        accuracy_score(y, pred),
        brier_score_loss(y, p),
    ])


def align_oof_runs(runs):
    """
    returns y (subjects,) e P (subjects, repetitions).
    """
    ids = pd.Index(runs[0]["y_idx"])
    if not ids.is_unique:
        raise ValueError("ID duplicati nella prima ripetizione")

    y = pd.Series(
        runs[0]["y_true_oof"], index=ids
    ).to_numpy()

    P = np.empty((len(ids), len(runs)), dtype=float)

    for r, run in enumerate(runs):
        run_ids = pd.Index(run["y_idx"])

        if (
            not run_ids.is_unique
            or len(run_ids) != len(ids)
            or not run_ids.difference(ids).empty
        ):
            raise ValueError(f"ID do not match, repetition {r}")

        y_r = pd.Series(
            run["y_true_oof"], index=run_ids
        ).reindex(ids).to_numpy()

        if not np.array_equal(y_r, y):
            raise ValueError(f"Results not coherent, repetition {r}")

        P[:, r] = pd.Series(
            run["y_prob_oof"], index=run_ids
        ).reindex(ids).to_numpy()

    if not np.isin(y, [0, 1]).all() or np.unique(y).size != 2:
        raise ValueError("Binary results expected 0/1 in both classes")
    if not np.isfinite(P).all() or ((P < 0) | (P > 1)).any():
        raise ValueError("Wrong probabilities")

    return y, P


def summarize_repeated_cv(
    runs, threshold=0.30, n_boot=1000, seed=42
):
    y, P = align_oof_runs(runs)
    rng = np.random.default_rng(seed)

    # One metric row for each CV repetition
    per_run = np.vstack([
        metric_values(y, P[:, r], threshold)
        for r in range(P.shape[1])
    ])

    mean = per_run.mean(axis=0)
    std = (
        per_run.std(axis=0, ddof=1)
        if P.shape[1] > 1
        else np.full(len(METRIC_NAMES), np.nan)
    )

    # Bootstrap: subject resampling, keeping joined predictions in all repetitions
    boot_means = []

    with tqdm(total=n_boot,initial=len(boot_means),desc="Bootstrap",unit="sample",) as pbar:
        while len(boot_means) < n_boot:
            idx = rng.integers(0, len(y), len(y))
            y_b = y[idx]

            # AUC e AUPRC require both classes
            if np.unique(y_b).size < 2:
                continue

            P_b = P[idx, :]
            scores_b = np.vstack([
                metric_values(y_b, P_b[:, r], threshold)
                for r in range(P_b.shape[1])
            ])
            boot_means.append(scores_b.mean(axis=0))
            pbar.update(1)
        
    ci_low, ci_high = np.percentile(
        np.asarray(boot_means), [2.5, 97.5], axis=0
    )

    summary = pd.DataFrame({
        "mean": mean,
        "std_between_runs": std,
        "ci95_low": ci_low,
        "ci95_high": ci_high,
    }, index=METRIC_NAMES)

    per_run = pd.DataFrame(
        per_run,
        columns=METRIC_NAMES,
        index=pd.Index(range(len(runs)), name="iteration"),
    )

    return summary, per_run

In [43]:
summaries = {}
scores_by_run = {}

for m in args.methods:
    meth = "+".join(m)
    runs = [results[meth][i] for i in range(args.iter)]

    summaries[meth], scores_by_run[meth] = summarize_repeated_cv(
        runs,
        threshold=0.30,
        n_boot=1000,
        seed=42,
    )

    print(f"\n{meth}")
    print(summaries[meth].round(3))
    summaries[meth].to_csv(f"{args.results_dir}/summary_{args.dataset}_{meth}_ITER{args.iter}.csv")

Bootstrap:   0%|          | 0/1000 [00:00<?, ?sample/s]


Med2Vec
              mean  std_between_runs  ci95_low  ci95_high
AUC          0.860             0.003     0.826      0.894
AUPRC        0.461             0.011     0.381      0.550
F1           0.412             0.031     0.338      0.480
PPV          0.509             0.033     0.419      0.601
Sensitivity  0.347             0.032     0.277      0.418
Specificity  0.964             0.004     0.955      0.973
NPV          0.933             0.003     0.919      0.944
MCC          0.370             0.031     0.294      0.444
Accuracy     0.905             0.004     0.891      0.918
Brier        0.068             0.000     0.060      0.078


Bootstrap:   0%|          | 0/1000 [00:00<?, ?sample/s]


Dipole
              mean  std_between_runs  ci95_low  ci95_high
AUC          0.853             0.006     0.818      0.885
AUPRC        0.458             0.024     0.383      0.542
F1           0.468             0.015     0.406      0.525
PPV          0.391             0.016     0.325      0.457
Sensitivity  0.585             0.041     0.514      0.651
Specificity  0.902             0.010     0.888      0.915
NPV          0.953             0.004     0.942      0.963
MCC          0.409             0.018     0.345      0.468
Accuracy     0.872             0.006     0.856      0.885
Brier        0.073             0.002     0.064      0.083


Bootstrap:   0%|          | 0/1000 [00:00<?, ?sample/s]


BiPadLSTM
              mean  std_between_runs  ci95_low  ci95_high
AUC          0.854             0.005     0.815      0.888
AUPRC        0.437             0.024     0.358      0.526
F1           0.480             0.011     0.416      0.541
PPV          0.387             0.014     0.325      0.454
Sensitivity  0.633             0.017     0.557      0.701
Specificity  0.893             0.008     0.878      0.908
NPV          0.958             0.002     0.946      0.967
MCC          0.426             0.012     0.360      0.488
Accuracy     0.868             0.006     0.851      0.883
Brier        0.069             0.002     0.061      0.078


Bootstrap:   0%|          | 0/1000 [00:00<?, ?sample/s]


GRU
              mean  std_between_runs  ci95_low  ci95_high
AUC          0.841             0.011     0.808      0.868
AUPRC        0.402             0.020     0.328      0.480
F1           0.372             0.048     0.308      0.428
PPV          0.483             0.047     0.398      0.568
Sensitivity  0.310             0.069     0.252      0.364
Specificity  0.964             0.010     0.956      0.971
NPV          0.929             0.006     0.915      0.941
MCC          0.334             0.037     0.267      0.394
Accuracy     0.901             0.004     0.887      0.913
Brier        0.077             0.003     0.067      0.089


## Save metric as dataframe

In [44]:
from sklearn.metrics import confusion_matrix

rows = []

for m in args.methods:
    meth = "+".join(m)
    summary = summaries[meth]

    row = {"model": meth}

    for metric in METRIC_NAMES:
        s = summary.loc[metric]
        row[metric] = (
            f"{s['mean']:.3f}±{s['std_between_runs']:.3f} "
            f"[{s['ci95_low']:.3f}-{s['ci95_high']:.3f}]"
        )

    # Una confusion matrix per iterazione, poi media elemento per elemento
    runs = [results[meth][i] for i in range(args.iter)]
    y, P = align_oof_runs(runs)

    matrices = []
    for i in range(args.iter):
        predicted = (P[:, i] >= 0.30).astype(int)

        # .T produce il formato richiesto: [[TP FP] [FN TN]]
        cm = confusion_matrix(
            y, predicted, labels=[0, 1]
        )
        matrices.append(cm)

    mean_cm = np.mean(matrices, axis=0)

    row["Confusion matrix"] = (
        f"[[{mean_cm[0, 0]:.1f} {mean_cm[0, 1]:.1f}] "
        f"[{mean_cm[1, 0]:.1f} {mean_cm[1, 1]:.1f}]]"
    )

    rows.append(row)

formatted_results = pd.DataFrame(rows)
formatted_results.set_index("model", inplace=True)
formatted_results.to_csv(f"{args.results_dir}/result_table_{args.dataset}_ITER{args.iter}.csv")
formatted_results

,AUC,AUPRC,F1,PPV,Sensitivity,Specificity,NPV,MCC,Accuracy,Brier,Confusion matrix
model,,,,,,,,,,,
Med2Vec,0.860±0.003 [0.826-0.894],0.461±0.011 [0.381-0.550],0.412±0.031 [0.338-0.480],0.509±0.033 [0.419-0.601],0.347±0.032 [0.277-0.418],0.964±0.004 [0.955-0.973],0.933±0.003 [0.919-0.944],0.370±0.031 [0.294-0.444],0.905±0.004 [0.891-0.918],0.068±0.000 [0.060-0.078],[[1167.8 43.2] [84.3 44.7]]
Dipole,0.853±0.006 [0.818-0.885],0.458±0.024 [0.383-0.542],0.468±0.015 [0.406-0.525],0.391±0.016 [0.325-0.457],0.585±0.041 [0.514-0.651],0.902±0.010 [0.888-0.915],0.953±0.004 [0.942-0.963],0.409±0.018 [0.345-0.468],0.872±0.006 [0.856-0.885],0.073±0.002 [0.064-0.083],[[1092.9 118.1] [53.5 75.5]]
BiPadLSTM,0.854±0.005 [0.815-0.888],0.437±0.024 [0.358-0.526],0.480±0.011 [0.416-0.541],0.387±0.014 [0.325-0.454],0.633±0.017 [0.557-0.701],0.893±0.008 [0.878-0.908],0.958±0.002 [0.946-0.967],0.426±0.012 [0.360-0.488],0.868±0.006 [0.851-0.883],0.069±0.002 [0.061-0.078],[[1081.4 129.6] [47.4 81.6]]
GRU,0.841±0.011 [0.808-0.868],0.402±0.020 [0.328-0.480],0.372±0.048 [0.308-0.428],0.483±0.047 [0.398-0.568],0.310±0.069 [0.252-0.364],0.964±0.010 [0.956-0.971],0.929±0.006 [0.915-0.941],0.334±0.037 [0.267-0.394],0.901±0.004 [0.887-0.913],0.077±0.003 [0.067-0.089],[[1167.1 43.9] [89.0 40.0]]


## To latex with split in N

In [45]:
args.dataset = "INA"
import pandas as pd
formatted_results = pd.read_csv(f"{args.results_dir}/result_table_{args.dataset}_ITER{args.iter}.csv", index_col=0)
formatted_results

,AUC,AUPRC,F1,PPV,Sensitivity,Specificity,NPV,MCC,Accuracy,Brier,Confusion matrix
model,,,,,,,,,,,
Med2Vec,0.860±0.003 [0.826-0.894],0.461±0.011 [0.381-0.550],0.412±0.031 [0.338-0.480],0.509±0.033 [0.419-0.601],0.347±0.032 [0.277-0.418],0.964±0.004 [0.955-0.973],0.933±0.003 [0.919-0.944],0.370±0.031 [0.294-0.444],0.905±0.004 [0.891-0.918],0.068±0.000 [0.060-0.078],[[1167.8 43.2] [84.3 44.7]]
Dipole,0.853±0.006 [0.818-0.885],0.458±0.024 [0.383-0.542],0.468±0.015 [0.406-0.525],0.391±0.016 [0.325-0.457],0.585±0.041 [0.514-0.651],0.902±0.010 [0.888-0.915],0.953±0.004 [0.942-0.963],0.409±0.018 [0.345-0.468],0.872±0.006 [0.856-0.885],0.073±0.002 [0.064-0.083],[[1092.9 118.1] [53.5 75.5]]
BiPadLSTM,0.854±0.005 [0.815-0.888],0.437±0.024 [0.358-0.526],0.480±0.011 [0.416-0.541],0.387±0.014 [0.325-0.454],0.633±0.017 [0.557-0.701],0.893±0.008 [0.878-0.908],0.958±0.002 [0.946-0.967],0.426±0.012 [0.360-0.488],0.868±0.006 [0.851-0.883],0.069±0.002 [0.061-0.078],[[1081.4 129.6] [47.4 81.6]]
GRU,0.841±0.011 [0.808-0.868],0.402±0.020 [0.328-0.480],0.372±0.048 [0.308-0.428],0.483±0.047 [0.398-0.568],0.310±0.069 [0.252-0.364],0.964±0.010 [0.956-0.971],0.929±0.006 [0.915-0.941],0.334±0.037 [0.267-0.394],0.901±0.004 [0.887-0.913],0.077±0.003 [0.067-0.089],[[1167.1 43.9] [89.0 40.0]]


In [46]:
import pandas as pd
from scripts.utils import highlight_best_values
nsplit = 4

# Divide le colonne in tre gruppi equilibrati
n = len(formatted_results.columns)
base, resto = divmod(n, nsplit)
dimensioni = [base + (i < resto) for i in range(nsplit)]

blocchi = []
inizio = 0
for dimensione in dimensioni:
    blocchi.append(formatted_results.iloc[:, inizio:inizio + dimensione].copy())
    inizio += dimensione

# Usa il primo blocco per definire le colonne della tabella finale
larghezza = max(dimensioni)
colonne_finali = (
    ["Model"]
    + blocchi[0].columns.tolist()
    + [""] * (larghezza - blocchi[0].shape[1])
)

pezzi = []
for i, blocco in enumerate(blocchi):
    if i > 0:
        intestazione = (
            ["Model"]
            + blocco.columns.tolist()
            + [""] * (larghezza - blocco.shape[1])
        )
        pezzi.append(pd.DataFrame([intestazione], columns=colonne_finali))

    righe = blocco.rename_axis("Model").reset_index()
    righe.columns = colonne_finali[:righe.shape[1]]
    righe = righe.reindex(columns=colonne_finali, fill_value="")
    pezzi.append(righe)

tabella = pd.concat(pezzi, ignore_index=True)
print(highlight_best_values(tabella.to_latex(index=False, escape=True)))

\begin{tabular}{llll}
\toprule
Model & AUC & AUPRC & F1 \\
\midrule
Med2Vec & \textbf{0.860}±0.003 [0.826-0.894] & \textbf{0.461}±0.011 [0.381-0.550] & 0.412±0.031 [0.338-0.480] \\
Dipole & 0.853±0.006 [0.818-0.885] & 0.458±0.024 [0.383-0.542] & 0.468±0.015 [0.406-0.525] \\
BiPadLSTM & 0.854±0.005 [0.815-0.888] & 0.437±0.024 [0.358-0.526] & \textbf{0.480}±0.011 [0.416-0.541] \\
GRU & 0.841±0.011 [0.808-0.868] & 0.402±0.020 [0.328-0.480] & 0.372±0.048 [0.308-0.428] \\
Model & PPV & Sensitivity & Specificity \\
Med2Vec & \textbf{0.509}±0.033 [0.419-0.601] & 0.347±0.032 [0.277-0.418] & \textbf{0.964}±0.004 [0.955-0.973] \\
Dipole & 0.391±0.016 [0.325-0.457] & 0.585±0.041 [0.514-0.651] & 0.902±0.010 [0.888-0.915] \\
BiPadLSTM & 0.387±0.014 [0.325-0.454] & \textbf{0.633}±0.017 [0.557-0.701] & 0.893±0.008 [0.878-0.908] \\
GRU & 0.483±0.047 [0.398-0.568] & 0.310±0.069 [0.252-0.364] & \textbf{0.964}±0.010 [0.956-0.971] \\
Model & NPV & MCC & Accuracy \\
Med2Vec & 0.933±0.003 [0.919-0.944] & 0.